In [1]:
import sys
sys.path.append(r"C:\traffic-demand-final\pipelining\src")
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from config import *

# Load train and test data
train = pd.read_csv(os.path.join(PROC_DIR, 'train_step02.csv'))
test = pd.read_csv(os.path.join(PROC_DIR, 'test_step02.csv'))

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (77299, 11)
Test shape: (41778, 10)


In [2]:
# Preserve original column for comparison
train_road_orig = train['RoadType'].copy()

# Determine order dynamically from config map
road_order = sorted(ROAD_TYPE_MAP.keys(), key=lambda k: ROAD_TYPE_MAP[k])
road_encoder = OrdinalEncoder(categories=[road_order])

# Fit only on train to prevent leakage
train['road_type_ord'] = road_encoder.fit_transform(train[['RoadType']])
test['road_type_ord'] = road_encoder.transform(test[['RoadType']])

print("Train road_type_ord value counts:")
print(train['road_type_ord'].value_counts())

compare_df = pd.DataFrame({'RoadType': train_road_orig, 'road_type_ord': train['road_type_ord']})
print("\nFirst 5 rows comparison:")
print(compare_df.head(5))

Train road_type_ord value counts:
road_type_ord
0.0    69830
1.0     3909
2.0     3560
Name: count, dtype: int64

First 5 rows comparison:
      RoadType  road_type_ord
0  Residential            0.0
1  Residential            0.0
2  Residential            0.0
3  Residential            0.0
4  Residential            0.0


Ordinal encoding was chosen over one-hot encoding for RoadType because there is a natural hierarchical order to the data. Bivariate analysis demonstrated that traffic demand monotonically increases from Residential to Street to Highway. Using one-hot encoding would eliminate this inherent relationship, while ordinal encoding properly preserves the sequence for tree-based machine learning models.

In [3]:
# Preserve original column for comparison
train_weather_orig = train['Weather'].copy()

# Determine order dynamically from config map
weather_order = sorted(WEATHER_MAP.keys(), key=lambda k: WEATHER_MAP[k])
weather_encoder = OrdinalEncoder(categories=[weather_order])

# Fit only on train to prevent leakage
train['weather_severity'] = weather_encoder.fit_transform(train[['Weather']])
test['weather_severity'] = weather_encoder.transform(test[['Weather']])

print("Train weather_severity value counts:")
print(train['weather_severity'].value_counts())

compare_weather_df = pd.DataFrame({'Weather': train_weather_orig, 'weather_severity': train['weather_severity']})
print("\nFirst 5 rows comparison:")
print(compare_weather_df.head(5))

Train weather_severity value counts:
weather_severity
0.0    28514
2.0    20824
1.0    20243
3.0     7718
Name: count, dtype: int64

First 5 rows comparison:
  Weather  weather_severity
0   Sunny               0.0
1   Sunny               0.0
2   Sunny               0.0
3   Rainy               2.0
4   Rainy               2.0


Bivariate EDA identified a consistent suppression hierarchy regarding weather conditions. Sunny weather serves as the clear baseline, with Foggy, Rainy, and Snowy conditions progressively suppressing overall traffic demand. We utilize ordinal encoding directly representing this hierarchy to mirror the true severity pattern and avoid inflating the feature space with unneeded binary columns.

In [4]:
# Apply direct dictionary mapping for binary features
train['LargeVehicles'] = train['LargeVehicles'].map(LARGE_VEHICLES_MAP)
test['LargeVehicles'] = test['LargeVehicles'].map(LARGE_VEHICLES_MAP)

train['Landmarks'] = train['Landmarks'].map(LANDMARKS_MAP)
test['Landmarks'] = test['Landmarks'].map(LANDMARKS_MAP)

print("LargeVehicles value counts:")
print(train['LargeVehicles'].value_counts())

print("\nLandmarks value counts:")
print(train['Landmarks'].value_counts())

print(f"\nNulls introduced in LargeVehicles: {train['LargeVehicles'].isnull().sum()}")
print(f"Nulls introduced in Landmarks: {train['Landmarks'].isnull().sum()}")

LargeVehicles value counts:
LargeVehicles
0    50673
1    26626
Name: count, dtype: int64

Landmarks value counts:
Landmarks
1    52042
0    25257
Name: count, dtype: int64

Nulls introduced in LargeVehicles: 0
Nulls introduced in Landmarks: 0


Binary encoding is strictly appropriate here since both features contain exactly two distinct states. Within our mapping, 0 represents the absence ('Not Allowed' or 'No') and 1 represents the presence ('Allowed' or 'Yes'). Mapping directly bypasses the unnecessary structural complexity associated with creating additional dummy variables.

In [5]:
# The day column operates as a binary label
day_orig_unique = train['day'].unique()

day_encoder = LabelEncoder()
train['day'] = day_encoder.fit_transform(train['day'])
test['day'] = day_encoder.transform(test['day'])

print(f"Unique day values before encoding: {day_orig_unique}")
print(f"Unique day values after encoding: {train['day'].unique()}")

Unique day values before encoding: [48 49]
Unique day values after encoding: [0 1]


Univariate analysis previously confirmed that the day column consists of just two distinct values across the entire dataset. Utilizing a standard LabelEncoder automatically maps these unique values directly to 0 and 1, making it the most mathematically efficient and correct approach for our model.

In [6]:
# Summary table of all encodings performed
summary_data = [
    {'Original Column': 'RoadType', 'New Column': 'road_type_ord', 'Unique Before': list(train_road_orig.unique()), 'Unique After': list(train['road_type_ord'].unique()), 'Nulls After': train['road_type_ord'].isnull().sum()},
    {'Original Column': 'Weather', 'New Column': 'weather_severity', 'Unique Before': list(train_weather_orig.unique()), 'Unique After': list(train['weather_severity'].unique()), 'Nulls After': train['weather_severity'].isnull().sum()},
    {'Original Column': 'LargeVehicles', 'New Column': 'LargeVehicles', 'Unique Before': ['Not Allowed', 'Allowed'], 'Unique After': list(train['LargeVehicles'].unique()), 'Nulls After': train['LargeVehicles'].isnull().sum()},
    {'Original Column': 'Landmarks', 'New Column': 'Landmarks', 'Unique Before': ['No', 'Yes'], 'Unique After': list(train['Landmarks'].unique()), 'Nulls After': train['Landmarks'].isnull().sum()},
    {'Original Column': 'day', 'New Column': 'day', 'Unique Before': list(day_orig_unique), 'Unique After': list(train['day'].unique()), 'Nulls After': train['day'].isnull().sum()}
]

summary_df = pd.DataFrame(summary_data)
print("Encoding Summary:")
print(summary_df.to_string())

print("\nData Types (confirming numeric):")
print(train[['road_type_ord', 'weather_severity', 'LargeVehicles', 'Landmarks', 'day']].dtypes)

Encoding Summary:
  Original Column        New Column                   Unique Before          Unique After  Nulls After
0        RoadType     road_type_ord  [Residential, Street, Highway]       [0.0, 1.0, 2.0]            0
1         Weather  weather_severity    [Sunny, Rainy, Foggy, Snowy]  [0.0, 2.0, 1.0, 3.0]            0
2   LargeVehicles     LargeVehicles          [Not Allowed, Allowed]                [0, 1]            0
3       Landmarks         Landmarks                       [No, Yes]                [0, 1]            0
4             day               day                        [48, 49]                [0, 1]            0

Data Types (confirming numeric):
road_type_ord       float64
weather_severity    float64
LargeVehicles         int64
Landmarks             int64
day                   int64
dtype: object


In [7]:
# Drop original categorical string columns
train = train.drop(columns=['RoadType', 'Weather'])
test = test.drop(columns=['RoadType', 'Weather'])

print("Train columns after dropping:")
print(train.columns.tolist())

Train columns after dropping:
['geohash', 'day', 'demand', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'hour', 'minute', 'road_type_ord', 'weather_severity']


In [8]:
# Save processed datasets
train_save_path = os.path.join(PROC_DIR, 'train_step03.csv')
test_save_path = os.path.join(PROC_DIR, 'test_step03.csv')

train.to_csv(train_save_path, index=False)
test.to_csv(test_save_path, index=False)

print(f"Saved train to: {train_save_path} (Shape: {train.shape})")
print(f"Saved test to: {test_save_path} (Shape: {test.shape})")

Saved train to: C:\traffic-demand-final\pipelining\processed\train_step03.csv (Shape: (77299, 11))
Saved test to: C:\traffic-demand-final\pipelining\processed\test_step03.csv (Shape: (41778, 10))


# Step 03 Complete — What Was Done

* Encoded RoadType into road_type_ord using sklearn OrdinalEncoder
* Encoded Weather into weather_severity using sklearn OrdinalEncoder
* Encoded LargeVehicles and Landmarks directly using pandas map
* Encoded day using sklearn LabelEncoder since it operates as a strict binary category
* Chose ordinal encoding over one-hot encoding for RoadType and Weather to preserve natural monotonic relationships
* Extracted specific encoding maps directly from the centralized config mappings
* Fitted all sklearn encoders strictly on the training dataset to prevent future data leakage
* Verified successful zero nulls and verified numeric outputs across all target columns
* Dropped original string categorical columns to prevent redundancies
* Saved processed dataset files as train_step03.csv and test_step03.csv
* The next step is Step 04: Geohash Target Encoding